# A Blurry Photo is Not the Original (CODA-CS)

**Paper Companion:** Section 1 (Definitions CS-D1, CS-D3) & Section 6 (Computational Companion).

A fine observation can be forgotten to a coarse one: take a sharp picture, then blur it. The composite $\mathrm{forget}\circ\mathrm{fine}$ is the coarse observation (the blurry photo). Two signals that match after blurring need not match before.

This notebook uses the shipped CODA-CS core calculator (`code/coda_cs_core.py`). All calculations use strictly **exact rational arithmetic** (fractions, not floating point numbers). Floats and booleans in numerical positions strictly raise `TypeError`.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Rational
import matplotlib.pyplot as plt

from coda_cs_core import (
    ContinuumQuery,
    ContinuumSignal,
    ProfileMap,
    ShadowSlice,
    ShadowTower,
    reproduce_worked_example,
    verify_bridge_admissibility,
)
print("CODA-CS core loaded successfully with exact rational arithmetic.")

CODA-CS core loaded successfully with exact rational arithmetic.


## 1. The Worked Example: Grid Quadrature on $X(t) = 2t$

Let a signal be $X(t) = 2t$ on the unit interval $[0, 1]$. We sample the signal on the uniform 5-node grid $\{0, 1/4, 1/2, 3/4, 1\}$.

The shipped left-Riemann quadrature query evaluates:
$$
\sum_{k=0}^3 X(t_k)^2 \cdot \Delta t
= 0^2\cdot\frac{1}{4} + \left(\frac{1}{2}\right)^2\cdot\frac{1}{4} + 1^2\cdot\frac{1}{4} + \left(\frac{3}{2}\right)^2\cdot\frac{1}{4}
= 0 + \frac{1}{16} + \frac{1}{4} + \frac{9}{16} = \frac{7}{8}.
$$
Any second signal that matches on these 5 nodes produces the exact same result $7/8$.

In [2]:
val1, val2, admissible = reproduce_worked_example()
print(f"Quadrature on signal 1: {val1}")
print(f"Quadrature on signal 2: {val2}")
print(f"Same sample profile => Same query answer? {val1 == val2}")
print(f"Admissible under 8-step gate? {admissible}")

assert val1 == Rational(7, 8)
assert val2 == Rational(7, 8)
assert admissible is True

Quadrature on signal 1: 7/8
Quadrature on signal 2: 7/8
Same sample profile => Same query answer? True
Admissible under 8-step gate? True


## 2. Fine vs. Coarse: Refinement and Forgetting

Now consider two observation structures (Definition CS-D3):
- **Fine observation:** 5 nodes $\{0, 1/4, 1/2, 3/4, 1\}$.
- **Coarse observation:** 3 nodes $\{0, 1/2, 1\}$.
- **Forgetful map:** $\mathrm{forget}: \mathbb{Q}^5 \to \mathbb{Q}^3$ keeps nodes $0, 2, 4$ and drops the odd quarters $1/4$ and $3/4$.

We construct two signals:
1. `sharp`: $X(t) = 2t$ at all nodes.
2. `twin`: Matches `sharp` at $t \in \{0, 1/2, 1\}$, but deviates drastically at the dropped nodes: $X(1/4) = 9$ and $X(3/4) = 8$.

In [3]:
fine_pts = (
    Rational(0, 4),
    Rational(1, 4),
    Rational(2, 4),
    Rational(3, 4),
    Rational(4, 4),
)
coarse_pts = (Rational(0, 1), Rational(1, 2), Rational(1, 1))

fine_prof = ProfileMap(profile_id="fine_grid", sample_points=fine_pts)
coarse_prof = ProfileMap(profile_id="coarse_grid", sample_points=coarse_pts)

def forget(fine_vec):
    return (fine_vec[0, 0], fine_vec[2, 0], fine_vec[4, 0])

sharp = ContinuumSignal(
    signal_id="sharp",
    samples={t: 2 * t for t in fine_pts},
)
twin = ContinuumSignal(
    signal_id="twin",
    samples={
        Rational(0, 4): Rational(0, 1),
        Rational(1, 4): Rational(9, 1),  # off-coarse spike
        Rational(2, 4): Rational(1, 1),
        Rational(3, 4): Rational(8, 1),  # off-coarse spike
        Rational(4, 4): Rational(2, 1),
    },
)

fine_sharp = fine_prof.project(sharp)
fine_twin = fine_prof.project(twin)

print("Fine profiles match? ", fine_sharp == fine_twin)
print("After forgetting, match?", forget(fine_sharp) == forget(fine_twin))
print("Coarse profiles match? ", coarse_prof.project(sharp) == coarse_prof.project(twin))

assert fine_sharp != fine_twin
assert forget(fine_sharp) == forget(fine_twin)
assert coarse_prof.project(sharp) == coarse_prof.project(twin)

Fine profiles match?  False
After forgetting, match? True
Coarse profiles match?  True


## 3. Visualizing the Blurry Photo

Notice how `sharp` and `twin` share the exact same coarse shadow (green markers), yet are completely different functions. The coarse shadow collapses them into the same fiber!

In [4]:
fig, ax = plt.subplots(figsize=(8, 4.5))
t_coarse_f = [float(t) for t in coarse_pts]
v_coarse_f = [float(sharp.evaluate_at(t)) for t in coarse_pts]
t_fine_f = [float(t) for t in fine_pts]
v_sharp_f = [float(sharp.evaluate_at(t)) for t in fine_pts]
v_twin_f = [float(twin.evaluate_at(t)) for t in fine_pts]

ax.plot(t_fine_f, v_sharp_f, 'o-', color='#2980b9', linewidth=2, label='sharp: X(t) = 2t')
ax.plot(t_fine_f, v_twin_f, 's--', color='#e67e22', linewidth=2, label='twin: spikes at 1/4 and 3/4')
ax.scatter(t_coarse_f, v_coarse_f, color='#27ae60', s=140, zorder=5, label='Coarse nodes {0, 1/2, 1}')
ax.set_xlabel('t', fontsize=11)
ax.set_ylabel('Signal value', fontsize=11)
ax.set_title('Coarse Shadow Collapses Different Signals into Same Fiber', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Query Evaluation and Refusal

- A query $q_{\mathrm{coarse}}$ that depends only on coarse nodes gives the **same** answer for both signals.
- A query $q_{\mathrm{quarter}}$ that queries $X(1/4)$ gives **different** answers.
- The CODA-CS verification pipeline detects this: evaluating $q_{\mathrm{quarter}}$ on the coarse tower fails fiber constancy and is refused with `R_not_fiber_constant`.

In [5]:
def coarse_energy(sig):
    tot = Rational(0, 1)
    for t in (Rational(0, 1), Rational(1, 2)):
        v = sig.evaluate_at(t)
        tot += v * v * Rational(1, 2)
    return tot

def uses_quarter(sig):
    return sig.evaluate_at(Rational(1, 4))

print(f"Coarse energy: sharp = {coarse_energy(sharp)}, twin = {coarse_energy(twin)}")
print(f"Query at 1/4:  sharp = {uses_quarter(sharp)}, twin = {uses_quarter(twin)}")

assert coarse_energy(sharp) == coarse_energy(twin)
assert uses_quarter(sharp) != uses_quarter(twin)

# Pass through the mechanical verification pipeline
tower = ShadowTower(
    tower_id="coarse_tower",
    slices=(ShadowSlice("S", 3, True),),
    is_cofinal=True,
)
q_bad = ContinuumQuery(query_id="at_quarter", degree_bound=1, evaluator=uses_quarter)
rep = verify_bridge_admissibility(
    query=q_bad,
    tower=tower,
    profile=coarse_prof,
    test_signals=[sharp, twin],
)
print(f"Is fiber constant on coarse tower? {rep.is_fiber_constant}")
print(f"Refusal code: {rep.refusal_code}")

assert rep.is_fiber_constant is False
assert rep.refusal_code == "R_not_fiber_constant"
print("VERIFIED: The coarse shadow cannot authorize queries that inspect forgotten fine detail.")

Coarse energy: sharp = 1/2, twin = 1/2
Query at 1/4:  sharp = 1/2, twin = 9
Is fiber constant on coarse tower? False
Refusal code: R_not_fiber_constant
VERIFIED: The coarse shadow cannot authorize queries that inspect forgotten fine detail.
